# SCIGEN Tutorial: Crystal Structure Generation with Structural Constraints

Generate crystal materials with targeted geometric patterns (kagome, honeycomb, triangular, etc.)
using the SCIGEN diffusion model.

**Paper:** [Structural constraint integration in a generative model for the discovery of quantum materials](https://doi.org/10.1038/s41563-025-02355-y) (Nature Materials, 2025)

This notebook demonstrates:
1. Setting up the SCIGEN environment in Google Colab
2. Loading a pretrained diffusion model
3. Generating crystal structures with structural constraints
4. Inspecting and visualizing the generated materials
5. Exporting results as CIF files

**Runtime:** Select **GPU** under Runtime > Change runtime type (T4 is sufficient).

---
## 1. Setup

### 1.1 (Optional) Google Drive for model caching

Mounting Google Drive lets you cache the pretrained model so you don't
have to re-download it each session. **Skip this cell** if you don't need caching.

In [ ]:
# Optional: Uncomment to mount Google Drive for persistent model caching
# from google.colab import drive
# drive.mount('/content/drive')
# CACHE_DIR = '/content/drive/MyDrive/scigen_models'

# If not using Drive, models live in the Colab runtime only
CACHE_DIR = None

### 1.2 Install dependencies

This installs PyTorch Geometric extensions (matching the Colab PyTorch/CUDA versions)
and the other libraries SCIGEN needs for generation.

> **Note:** The PyG extensions (torch-scatter, torch-sparse, torch-cluster) install
> quickly (~1-2 min) when prebuilt wheels are available. If wheels are not found for
> your exact PyTorch+CUDA version, they compile from source which takes ~10 min.
> This is a one-time cost per Colab session.

In [ ]:
import torch, os

# Detect PyTorch and CUDA versions for compatible wheel installation
# Truncate to major.minor (e.g., 2.5.1 -> 2.5.0) to match PyG wheel index
_ver_parts = torch.__version__.split('+')[0].split('.')
TORCH_VER = f'{_ver_parts[0]}.{_ver_parts[1]}.0'
CUDA_VER = 'cu' + torch.version.cuda.replace('.', '')
PYG_WHEEL_URL = f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_VER}.html'
os.environ['PYG_WHEEL_URL'] = PYG_WHEEL_URL
print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda}')
print(f'PyG wheel index: {PYG_WHEEL_URL}')

In [ ]:
%%time
# Install PyG extensions from prebuilt wheels (~1-2 min with wheels, ~10 min if compiled from source)
!pip install -q torch-scatter torch-sparse torch-cluster \
    -f $PYG_WHEEL_URL

# Install remaining dependencies
!pip install -q torch-geometric 'lightning>=2.0' 'hydra-core>=1.2,<1.4' omegaconf \
    python-dotenv pymatgen smact einops p-tqdm pyxtal pathos

print('All dependencies installed.')

### 1.3 Clone the SCIGEN repository

In [ ]:
REPO_URL = 'https://github.com/RyotaroOKabe/APS_demo_SCIGEN.git'
PROJECT_DIR = '/content/APS_demo_SCIGEN'

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}

print(f'Repository cloned to {PROJECT_DIR}')

### 1.4 Configure environment

SCIGEN reads `PROJECT_ROOT`, `HYDRA_JOBS`, and `WANDB_DIR` from the environment
at import time. These **must** be set before any `import scigen` statement.

In [ ]:
import os, sys

PROJECT_DIR = '/content/APS_demo_SCIGEN'

# Required environment variables (must be set before importing scigen)
os.environ['PROJECT_ROOT'] = PROJECT_DIR
os.environ['HYDRA_JOBS'] = PROJECT_DIR
os.environ['WANDB_DIR'] = os.path.join(PROJECT_DIR, 'wandb')
os.makedirs(os.environ['WANDB_DIR'], exist_ok=True)

# Add project root and script/ to Python path
sys.path.insert(0, PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, 'script'))

# Change to project root (gen_utils loads ./data/kde_bond.pkl relative to CWD)
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

---
## 2. Download Pretrained Model

The pretrained SCIGEN model weights are hosted on
[Figshare](https://doi.org/10.6084/m9.figshare.27778134).
This cell uses the Figshare REST API to download them reliably
(the bulk `ndownloader` URL often fails with redirects).

In [ ]:
import shutil, json, zipfile
from pathlib import Path
from urllib.request import urlopen, urlretrieve

MODEL_DIR = Path(PROJECT_DIR) / 'models'
FIGSHARE_API_URL = 'https://api.figshare.com/v2/articles/27778134'

# Use Google Drive cache if available
if CACHE_DIR is not None:
    cache_path = Path(CACHE_DIR)
    if (cache_path / 'mp_20').exists() and list((cache_path / 'mp_20').glob('*.ckpt')):
        print(f'Using cached model from Google Drive: {cache_path / "mp_20"}')
        MODEL_DIR.mkdir(parents=True, exist_ok=True)
        if not (MODEL_DIR / 'mp_20').exists():
            shutil.copytree(str(cache_path / 'mp_20'), str(MODEL_DIR / 'mp_20'))

# Download if not already present
need_download = not (MODEL_DIR / 'mp_20').exists() or not list((MODEL_DIR / 'mp_20').glob('*.ckpt'))

if need_download:
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    # Query Figshare API for file list
    print('Querying Figshare API...')
    article = json.loads(urlopen(FIGSHARE_API_URL).read().decode())
    files = article.get('files', [])
    print(f'Found {len(files)} file(s):')
    for f in files:
        print(f'  {f["name"]}  ({f.get("size", 0) / 1e6:.1f} MB)')

    # Download each file
    for f in files:
        dest = MODEL_DIR / f['name']
        print(f'Downloading {f["name"]}...')
        urlretrieve(f['download_url'], str(dest))
        print(f'  Saved ({dest.stat().st_size / 1e6:.1f} MB)')

        # Extract zip files
        if dest.suffix == '.zip':
            print(f'  Extracting...')
            with zipfile.ZipFile(dest, 'r') as zf:
                zf.extractall(MODEL_DIR)
            dest.unlink()

    # Find model files (hparams.yaml) and move to models/mp_20/ if needed
    if not (MODEL_DIR / 'mp_20').exists() or not list((MODEL_DIR / 'mp_20').glob('*.ckpt')):
        found = list(MODEL_DIR.rglob('hparams.yaml'))
        if found:
            src_dir = found[0].parent
            dest_dir = MODEL_DIR / 'mp_20'
            if src_dir.resolve() != dest_dir.resolve():
                print(f'Moving model: {src_dir.name}/ -> mp_20/')
                if dest_dir.exists():
                    shutil.rmtree(dest_dir)
                shutil.move(str(src_dir), str(dest_dir))

    # Cache to Google Drive if mounted
    if CACHE_DIR is not None and (MODEL_DIR / 'mp_20').exists():
        cache_path = Path(CACHE_DIR)
        cache_path.mkdir(parents=True, exist_ok=True)
        if not (cache_path / 'mp_20').exists():
            shutil.copytree(str(MODEL_DIR / 'mp_20'), str(cache_path / 'mp_20'))
            print(f'Model cached to Google Drive: {cache_path / "mp_20"}')

# Verify
MODEL_PATH = MODEL_DIR / 'mp_20'
print(f'\nModel path: {MODEL_PATH}')
if MODEL_PATH.exists():
    contents = [f.name for f in sorted(MODEL_PATH.iterdir())]
    print(f'Contents: {contents}')
    ckpts = list(MODEL_PATH.glob('*.ckpt'))
    assert ckpts, f'ERROR: No .ckpt files in {MODEL_PATH}. Download may have failed.'
    print(f'Checkpoints: {[c.name for c in ckpts]}')
else:
    raise FileNotFoundError(
        f'{MODEL_PATH} does not exist!\n'
        f'The Figshare download may have failed. Try:\n'
        f'  1. Re-run this cell\n'
        f'  2. Manually download from https://doi.org/10.6084/m9.figshare.27778134\n'
        f'     and extract to {MODEL_PATH}'
    )

---
## 3. Load the Pretrained Model

We load the SCIGEN diffusion model using Hydra for configuration
and manual state-dict loading for compatibility with any
PyTorch Lightning version.

In [ ]:
import torch
import numpy as np
import hydra
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from pathlib import Path


def load_model_for_inference(model_path, device='cpu'):
    """Load SCIGEN pretrained model for inference.

    Uses manual state_dict loading instead of pytorch_lightning's
    load_from_checkpoint, so it works with any PL version.
    """
    model_path = Path(model_path)

    # Clear previous hydra state (allows re-running this cell)
    GlobalHydra.instance().clear()

    # Load model config from hparams.yaml
    with initialize_config_dir(config_dir=str(model_path.resolve()), version_base=None):
        cfg = compose(config_name='hparams')

    # Instantiate model architecture (empty weights)
    model = hydra.utils.instantiate(
        cfg.model, optim=cfg.optim, data=cfg.data,
        logging=cfg.logging, _recursive_=False,
    )

    # Find checkpoint file
    ckpts = sorted(model_path.glob('*.ckpt'))
    if not ckpts:
        raise FileNotFoundError(f'No .ckpt files found in {model_path}')

    # Prefer *-last.ckpt, otherwise take the last sorted file
    ckpt_path = next((c for c in ckpts if 'last' in c.name), ckpts[-1])
    print(f'Loading checkpoint: {ckpt_path.name}')

    # Load weights manually (avoids PL version issues)
    checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['state_dict'], strict=False)

    # Load data scalers
    for attr, fname in [('lattice_scaler', 'lattice_scaler.pt'),
                        ('scaler', 'prop_scaler.pt')]:
        fpath = model_path / fname
        if fpath.exists():
            setattr(model, attr, torch.load(fpath, map_location='cpu', weights_only=False))

    model = model.to(device)
    model.eval()
    return model, cfg

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model, cfg = load_model_for_inference(MODEL_PATH, device=device)

# Attach the generation sampling method
from scigen.pl_modules.diffusion_w_type import sample_scigen
model.sample_scigen = sample_scigen.__get__(model)

num_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded: {num_params:,} parameters')

---
## 4. Generate Crystal Structures

Configure generation parameters and run the diffusion model.

**Available structural constraints:**

| Code | Lattice | Known atoms |
|------|---------|-------------|
| `tri` | Triangular | 1 |
| `hon` | Honeycomb | 2 |
| `kag` | Kagome | 3 |
| `sqr` | Square | 1 |
| `elt` | Elongated triangular | 2 |
| `sns` | Snub square | 4 |
| `tsq` | Truncated square | 4 |
| `van` | No constraint (vanilla) | 0 |

Try changing `SC_TYPE`, `ELEMENT`, and `BATCH_SIZE` below!

In [ ]:
from torch_geometric.data import DataLoader
from gen_utils import SampleDataset

# ---- Generation parameters (edit these!) ----
SC_TYPE = 'kag'       # Structural constraint type
ELEMENT = 'Mn'        # Element for constrained sites
BATCH_SIZE = 4        # Structures per batch (keep small for Colab)
NUM_BATCHES = 1       # Number of batches
FRAC_Z = 0.5          # z-coordinate of the 2D pattern layer
STEP_LR = 5e-6        # Diffusion step size (recommended for mp_20)
SEED = 42             # Random seed for reproducibility

# Atom count ranges per structural constraint
SC_NATM_RANGE = {
    'tri': [1, 4],  'hon': [1, 8],  'kag': [1, 12],
    'sqr': [1, 4],  'elt': [1, 8],  'sns': [1, 16],
    'tsq': [1, 16], 'van': [1, 20],
}

total_structures = BATCH_SIZE * NUM_BATCHES
natm_range = SC_NATM_RANGE.get(SC_TYPE, [1, 20])

print(f'Will generate {total_structures} structures')
print(f'  Lattice type: {SC_TYPE}')
print(f'  Element: {ELEMENT}')
print(f'  Atoms per unit cell: {natm_range[0]}-{natm_range[1]}')

In [ ]:
import time

# Build the sample dataset with structural constraints
test_set = SampleDataset(
    dataset='mp_20',
    natm_range=natm_range,
    total_num=total_structures,
    bond_sigma_per_mu=None,
    use_min_bond_len=False,
    known_species=[ELEMENT],
    sc_list=[SC_TYPE],
    frac_z=FRAC_Z,
    c_vec_cons={'scale': None, 'vert': False},
    reduced_mask=False,
    seed=SEED,
    device=device,
)

test_loader = DataLoader(test_set, batch_size=BATCH_SIZE)

# Run diffusion sampling
all_frac_coords = []
all_atom_types = []
all_lattices = []
all_num_atoms = []
all_num_known = []

print(f'Running diffusion (step_lr={STEP_LR})...')
start_time = time.time()

for idx, batch in enumerate(test_loader):
    if torch.cuda.is_available():
        batch.cuda()
    outputs, traj = model.sample_scigen(batch, step_lr=STEP_LR)

    all_frac_coords.append(outputs['frac_coords'].detach().cpu())
    # atom_types from diffusion are logits (N, 100) -> convert to indices
    raw_types = outputs['atom_types'].detach().cpu()
    if raw_types.dim() == 2:
        raw_types = raw_types.argmax(dim=-1) + 1  # 1-indexed element numbers
    all_atom_types.append(raw_types)
    all_lattices.append(outputs['lattices'].detach().cpu())
    all_num_atoms.append(outputs['num_atoms'].detach().cpu())
    all_num_known.append(outputs['num_known'].detach().cpu())

    print(f'  Batch {idx + 1}/{len(test_loader)} complete')

elapsed = time.time() - start_time
print(f'\nGeneration complete in {elapsed:.1f}s ({elapsed / total_structures:.1f}s per structure)')

# Concatenate results
frac_coords = torch.cat(all_frac_coords, dim=0)
atom_types = torch.cat(all_atom_types, dim=0)
lattices = torch.cat(all_lattices, dim=0)
num_atoms = torch.cat(all_num_atoms, dim=0)
num_known = torch.cat(all_num_known, dim=0)

---
## 5. Inspect Results

Convert raw tensor outputs to crystal structures and examine the generated materials.

In [ ]:
def lattices_to_params(lattices):
    """Convert 3x3 lattice matrices to (lengths, angles) parameterization."""
    lengths = torch.sqrt(torch.sum(lattices ** 2, dim=-1))
    angles = torch.zeros_like(lengths)
    for i in range(3):
        j, k = (i + 1) % 3, (i + 2) % 3
        cos_angle = torch.clamp(
            torch.sum(lattices[..., j, :] * lattices[..., k, :], dim=-1)
            / (lengths[..., j] * lengths[..., k]),
            -1.0, 1.0,
        )
        angles[..., i] = torch.arccos(cos_angle) * 180.0 / np.pi
    return lengths, angles


lengths, angles = lattices_to_params(lattices)

# Element symbol lookup
from scigen.common.data_utils import chemical_symbols
from collections import Counter

print(f'Generated {num_atoms.shape[0]} structures\n')
print(f'{"#":>3} {"Atoms":>5} {"Known":>5} {"Composition":<25} {"a":>7} {"b":>7} {"c":>7}  {"alpha":>6} {"beta":>6} {"gamma":>6}')
print('-' * 95)

start_idx = 0
for i in range(num_atoms.shape[0]):
    na = int(num_atoms[i])
    nk = int(num_known[i]) if i < len(num_known) else 0
    types_i = atom_types[start_idx:start_idx + na].int().tolist()
    symbols = [chemical_symbols[t] for t in types_i]
    comp = Counter(symbols)
    comp_str = ' '.join(f'{el}{n}' for el, n in sorted(comp.items()))
    l = lengths[i].tolist()
    a = angles[i].tolist()
    print(f'{i:3d} {na:5d} {nk:5d} {comp_str:<25} {l[0]:7.2f} {l[1]:7.2f} {l[2]:7.2f}  {a[0]:6.1f} {a[1]:6.1f} {a[2]:6.1f}')
    start_idx += na

In [ ]:
# Convert to pymatgen Structure objects
from pymatgen.core.structure import Structure
from pymatgen.core.lattice import Lattice

structures = []
start_idx = 0

for i in range(num_atoms.shape[0]):
    na = int(num_atoms[i])
    types_i = atom_types[start_idx:start_idx + na].int().tolist()
    coords_i = frac_coords[start_idx:start_idx + na].numpy()
    lengths_i = lengths[i].tolist()
    angles_i = angles[i].tolist()

    species = [chemical_symbols[t] for t in types_i]
    lattice = Lattice.from_parameters(*lengths_i, *angles_i)

    try:
        structure = Structure(lattice, species, coords_i, coords_are_cartesian=False)
        structures.append(structure)
        print(f'Structure {i}: {structure.composition.reduced_formula}'
              f'  ({na} atoms, V={structure.volume:.1f} A^3)')
    except Exception as e:
        print(f'Structure {i}: construction failed - {e}')

    start_idx += na

print(f'\nSuccessfully constructed {len(structures)}/{num_atoms.shape[0]} structures')

---
## 6. Visualize Structures

Simple 3D scatter plots of the generated unit cells.

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D


def plot_structure_3d(structure, title='', ax=None):
    """Simple 3D scatter plot of atom positions in a crystal structure."""
    if ax is None:
        fig = plt.figure(figsize=(6, 5))
        ax = fig.add_subplot(111, projection='3d')

    cart_coords = structure.cart_coords
    species = [str(s.specie) for s in structure]

    # Color atoms by element
    unique_el = sorted(set(species))
    cmap = plt.cm.Set1(np.linspace(0, 0.9, max(len(unique_el), 1)))
    color_map = {el: cmap[i] for i, el in enumerate(unique_el)}

    for el in unique_el:
        mask = [s == el for s in species]
        coords = cart_coords[mask]
        ax.scatter(
            coords[:, 0], coords[:, 1], coords[:, 2],
            s=200, c=[color_map[el]], label=el,
            alpha=0.8, edgecolors='k', linewidths=0.5,
        )

    ax.set_xlabel('x (A)')
    ax.set_ylabel('y (A)')
    ax.set_zlabel('z (A)')
    ax.legend(loc='upper left', fontsize=8)
    ax.set_title(title, fontsize=10)
    return ax


n_plot = min(4, len(structures))
fig = plt.figure(figsize=(5 * n_plot, 5))

for i in range(n_plot):
    ax = fig.add_subplot(1, n_plot, i + 1, projection='3d')
    formula = structures[i].composition.reduced_formula
    plot_structure_3d(structures[i], title=f'#{i}: {formula}', ax=ax)

plt.tight_layout()
plt.show()

---
## 7. (Optional) Export as CIF Files

Save generated structures as CIF files and download them.

In [ ]:
output_dir = os.path.join(PROJECT_DIR, 'generated_cifs')
os.makedirs(output_dir, exist_ok=True)

for i, structure in enumerate(structures):
    formula = structure.composition.reduced_formula
    cif_path = os.path.join(output_dir, f'{SC_TYPE}_{formula}_{i:03d}.cif')
    structure.to(filename=cif_path, fmt='cif')
    print(f'Saved: {os.path.basename(cif_path)}')

print(f'\n{len(structures)} CIF files saved to {output_dir}')

In [ ]:
# Download CIF files as a zip archive (Colab only)
try:
    import shutil
    from google.colab import files
    zip_name = f'scigen_{SC_TYPE}_{ELEMENT}'
    zip_path = shutil.make_archive(
        os.path.join(PROJECT_DIR, zip_name), 'zip', output_dir
    )
    files.download(zip_path)
    print(f'Downloading {zip_name}.zip')
except ImportError:
    print('Not running in Colab - skipping download.')
except Exception as e:
    print(f'Download failed: {e}')
    print(f'CIF files are available at: {output_dir}')

---
## Next Steps

- **Try different lattice types:** Change `SC_TYPE` to `'hon'`, `'tri'`, `'sqr'`, etc.
- **Try different elements:** Change `ELEMENT` to `'Fe'`, `'Co'`, `'Ni'`, `'Ti'`, etc.
- **Generate more structures:** Increase `BATCH_SIZE` or `NUM_BATCHES`.
- **Custom constraints:** See the [README](https://github.com/RyotaroOKabe/APS_demo_SCIGEN#create-your-own-structural-constraint) for how to define your own lattice types.
- **Pre-screening:** Use `script/eval_screen.py` to filter structures by stability criteria.

For more details, see the [SCIGEN paper](https://doi.org/10.1038/s41563-025-02355-y).